# Exploratory Data Analysis — Data Quality Analysis

Member 3 - Mathuran: Data Quality section of `02_eda.ipynb`.

Dataset: Kaggle Playground Series S4E2 — Obesity Risk Prediction (https://www.kaggle.com/competitions/playground-series-s4e2/data). Raw file: `data/raw/obesity.csv`. The raw dataset is inspected without modifying it.

## Objectives (Member 3 scope only)

- Analyse missing values
- Check duplicate records
- Verify data types
- Analyse statistical properties
- Identify dataset issues requiring preprocessing and state how each should be handled

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler,
)

In [ ]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "obesity.csv"
)

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_PATH)
print("Dataset exists:", DATA_PATH.exists())

In [ ]:
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully")
print("Dataset shape:", df.shape)

In [ ]:
print("Columns:", df.columns.tolist())

In [ ]:
identifier_column = "id"
target_column = "NObeyesdad"

print("Identifier column:", identifier_column)
print("Target column:", target_column)

## 1. Missing values

Every cell is checked for `NaN`. The count and percentage per column show whether any imputation is required before preprocessing.

In [ ]:
missing_counts = df.isnull().sum()
missing_percentages = (df.isnull().mean().mul(100).round(2))
missing_summary = pd.DataFrame({"Missing count": missing_counts, "Missing percentage": missing_percentages})
print("Total missing cells:", int(missing_counts.sum()))
display(missing_summary)

## 2. Duplicate records

Exact duplicate rows would inflate the sample size, and a duplicated `id` would break the identifier assumption. Both are checked.

In [ ]:
exact_duplicates = int(df.duplicated().sum())
feature_duplicates = int(df.duplicated(subset=[c for c in df.columns if c != identifier_column]).sum())
print("Exact duplicate rows:", exact_duplicates)
print("Duplicate rows excluding id:", feature_duplicates)
print("id is unique:", bool(df[identifier_column].is_unique))
print("id unique count:", int(df[identifier_column].nunique()))

## 3. Data types and statistical properties

Pandas dtypes are compared against the expected schema, then `describe()`, skewness, kurtosis and the IQR rule quantify centre, spread and shape.

In [ ]:
print(df.dtypes)

In [ ]:
numerical_features = ["Age", "Height", "Weight", "FCVC", "NCP", "CH2O", "FAF", "TUE"]
display(df[numerical_features].describe().T.round(2))

In [ ]:
shape_summary = pd.DataFrame({"Skewness": df[numerical_features].skew().round(3), "Kurtosis": df[numerical_features].kurt().round(3)})
display(shape_summary)

In [ ]:
q1 = df[numerical_features].quantile(0.25)
q3 = df[numerical_features].quantile(0.75)
iqr = q3 - q1
lo = q1 - 1.5 * iqr
hi = q3 + 1.5 * iqr
oc = ((df[numerical_features] < lo) | (df[numerical_features] > hi)).sum()
op = (oc / len(df) * 100).round(2)
display(pd.DataFrame({"IQR outlier count": oc, "IQR outlier percentage": op}))

In [ ]:
figure, axes = plt.subplots(nrows=2, ncols=4, figsize=(14, 7))
for axis, column in zip(axes.flat, numerical_features):
    sns.boxplot(data=df, x=column, ax=axis)
    axis.set_title(f"Boxplot of {column}")
    axis.set_xlabel(column)
plt.tight_layout()
plt.show()

In [ ]:
X = df.drop(columns=[identifier_column, target_column])
y = df[target_column].copy()
print("Original dataset shape:", df.shape)
print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)

In [ ]:
print("Identifier present in X:", identifier_column in X.columns)
print("Target present in X:", target_column in X.columns)

In [ ]:
print("Number of target classes:", y.nunique())
print("Target classes:")
for class_name in sorted(y.unique()):
    print(class_name)

In [ ]:
numerical_features = ["Age", "Height", "Weight", "FCVC", "NCP", "CH2O", "FAF", "TUE"]
ordinal_features = ["CAEC", "CALC"]
nominal_features = ["Gender", "family_history_with_overweight", "FAVC", "SMOKE", "SCC", "MTRANS"]
print("Numerical features:")
print(numerical_features)
print("\nOrdinal categorical features:")
print(ordinal_features)
print("\nNominal categorical features:")
print(nominal_features)

## 4. Dataset issues requiring preprocessing (Member 3 reasoning)

Each technique below is justified by a data-quality finding: median imputation plus scaling for skewed numericals, most-frequent imputation plus ordinal encoding for ordered categories, and most-frequent imputation plus one-hot encoding for nominal categories including rare levels.

In [ ]:
caec_order = ["no", "Sometimes", "Frequently", "Always"]
calc_order = ["no", "Sometimes", "Frequently"]
ordinal_category_orders = {"CAEC": caec_order, "CALC": calc_order}
ordinal_category_orders

In [ ]:
numerical_pipeline = Pipeline(steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
numerical_pipeline

In [ ]:
ordinal_categories = [ordinal_category_orders[c] for c in ordinal_features]
ordinal_pipeline = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OrdinalEncoder(categories=ordinal_categories, handle_unknown="use_encoded_value", unknown_value=-1))])
ordinal_pipeline

In [ ]:
nominal_pipeline = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
nominal_pipeline

## 5. Leakage-safe stratified split and combined preprocessor

The split happens before any fitting, keeps all 7 classes proportional with `stratify`, and every transformer is fitted on training data only. Dropping `id` and the target from `X` removes identifier and target leakage.

In [ ]:
RANDOM_STATE = 42
TRAIN_SIZE = 0.70
TEMPORARY_SIZE = 0.30
print("Total split proportion:", TRAIN_SIZE + TEMPORARY_SIZE)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, train_size=TRAIN_SIZE, test_size=TEMPORARY_SIZE, random_state=RANDOM_STATE, stratify=y)
print("Training features:", X_train.shape)
print("Temporary features:", X_temp.shape)

In [ ]:
X_validation, X_test, y_validation, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp)
print("Training features:", X_train.shape)
print("Validation features:", X_validation.shape)
print("Test features:", X_test.shape)

In [ ]:
preprocessor = ColumnTransformer(transformers=[("numerical", numerical_pipeline, numerical_features), ("ordinal", ordinal_pipeline, ordinal_features), ("nominal", nominal_pipeline, nominal_features)], remainder="drop", verbose_feature_names_out=True)
preprocessor

In [ ]:
X_train_transformed = preprocessor.fit_transform(X_train)
X_validation_transformed = preprocessor.transform(X_validation)
X_test_transformed = preprocessor.transform(X_test)
print("Training transformed shape:", X_train_transformed.shape)
print("Validation transformed shape:", X_validation_transformed.shape)
print("Test transformed shape:", X_test_transformed.shape)

In [ ]:
transformed_feature_names = (preprocessor.get_feature_names_out())
print("Number of transformed features:", len(transformed_feature_names))
transformed_feature_names

## 6. Data-quality conclusion and handoff (Member 3)

- Missing: 0 NaN everywhere; median/most-frequent imputers kept as safeguards.
- Duplicates: 0 rows; `id` unique; `id` and target excluded from `X`.
- Types: int64 / float64 / strings as expected; no parsing repairs.
- Shape: Age skew 1.59, NCP skew -1.56; median + StandardScaler, no deletion.
- Categories: CAEC/CALC use fixed orders with unknown_value -1; nominals one-hot with ignore.
- Split: 70/15/15 stratified; transformers fit on train only, so no leakage.
